In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB 

# Building a SMS spam detector

In [2]:
# load the dataset
url = 'https://raw.githubusercontent.com/um-perez-alvaro/Data-Science-Practice/master/Data/sms.tsv.txt'
sms = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

In [3]:
sms.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# spam example
print(sms[sms.label=='spam'].message.iloc[100])

To review and KEEP the fantastic Nokia N-Gage game deck with Club Nokia, go 2 www.cnupdates.com/newsletter. unsubscribe from alerts reply with the word OUT


In [5]:
# ham example
print(sms[sms.label=='ham'].message.iloc[100])

Hmm...my uncle just informed me that he's paying the school directly. So pls buy food.


In [6]:
sms.label.value_counts()

ham     4825
spam     747
Name: label, dtype: int64

In [7]:
# feature matrix/target vector
X = sms.message
y = sms.label

In [8]:
# train/test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y)

In [9]:
# initialize the vectorizer (with default parameters)
vect = CountVectorizer(stop_words='english',max_features=1000,min_df=10)

In [10]:
# learn training vocabulary, then use it to create a document-term matrix
vect.fit(X_train)
X_train_dtm = vect.transform(X_train)

In [11]:
# transform testing data (using fitted vocabulary) into a document-term matrix
X_test_dtm = vect.transform(X_test)

## Naive Bayes model

In [12]:
# import and initialize a Multinomial Naive Bayes model
from sklearn.naive_bayes import MultinomialNB
nb_clf = MultinomialNB()

In [13]:
# train the model using X_train_dtm 
nb_clf.fit(X_train_dtm, y_train)

MultinomialNB()

In [14]:
# make class predictions for X_test_dtm
y_test_pred = nb_clf.predict(X_test_dtm)

In [15]:
# evaluate the model
from sklearn.metrics import accuracy_score, confusion_matrix

In [16]:
# accuracy
accuracy_score(list(y_test), y_test_pred)

0.9798994974874372

In [17]:
# confusion matrix
confusion_matrix(y_test, y_test_pred)

array([[1183,   15],
       [  13,  182]])

In [18]:
# print messages text for the false positives (ham incorrectly classified as spam) 
X_test[(y_test=='ham') & (y_test_pred=='spam')]

2340    Cheers for the message Zogtorius. Ive been st...
835                            Surely result will offer:)
5159                         No but the bluray player can
5475    Dhoni have luck to win some big title.so we wi...
1988                     No calls..messages..missed calls
3797    They have a thread on the wishlist section of ...
1559    Message from . I am at Truro Hospital on ext. ...
3415                              No pic. Please re-send.
75            I am waiting machan. Call me once you free.
5046    We have sent JD for Customer Service cum Accou...
495                      Are you free now?can i call now?
4894                               Send me the new number
4702                               I liked the new mobile
4862                               Nokia phone is lovly..
511     8 at the latest, g's still there if you can sc...
Name: message, dtype: object

In [19]:
X_test[3797]

'They have a thread on the wishlist section of the forums where ppl post nitro requests. Start from the last page and collect from the bottom up.'

In [20]:
# print messages text for the false negatives (span incorrectly classified as ham)  
X_test[(y_test=='spam') & (y_test_pred=='ham')]

4821    Check Out Choose Your Babe Videos @ sms.shsex....
4213    Missed call alert. These numbers called but le...
3530    Xmas & New Years Eve tickets are now on sale f...
2269                    88066 FROM 88066 LOST 3POUND HELP
1638    0A$NETWORKS allow companies to bill for SMS, s...
1196    You have 1 new voicemail. Please call 08719181503
2402    Babe: U want me dont u baby! Im nasty and have...
3501    Dorothy@kiefer.com (Bank of Granite issues Str...
2558    This message is brought to you by GMW Ltd. and...
5370    dating:i have had two of these. Only started a...
2295     You have 1 new message. Please call 08718738034.
869     Hello. We need some posh birds and chaps to us...
68      Did you hear about the new "Divorce Barbie"? I...
Name: message, dtype: object

In [21]:
# example of false negatives
X_test[4514]

KeyError: 4514

## How does Naive Bayes choose between spam and ham

In [29]:
# store the vocabulary of X_train
words = vect.get_feature_names_out()

In [30]:
len(words)

643

In [31]:
# Naive Bayes counts the number of times each word appears in each class
# Rows represent classes (ham and spam), columns represent words
nb_clf.feature_count_

array([[ 0.,  0.,  0., ..., 22.,  3., 32.],
       [20., 11., 12., ...,  3.,  9.,  0.]])

In [32]:
nb_clf.classes_

array(['ham', 'spam'], dtype='<U4')

In [33]:
# number of times each word appears across all ham messages
ham_word_count = nb_clf.feature_count_[0,:]
# number of times each word appears across all spam messages
spam_word_count = nb_clf.feature_count_[1,:]

In [34]:
# create a DataFrame of words with their separate ham and spam counts
words = pd.DataFrame({'word' : words, 'ham' : ham_word_count, 'spam' : spam_word_count}).set_index('word')
words.head()

,ham,spam
word,,
000,0.0,20.0
03,0.0,11.0
08000930705,0.0,12.0
10,8.0,22.0
100,1.0,32.0


In [35]:
# add 1 to the columns counts to avoid dividing by 0
words.ham = words.ham+1
words.spam = words.spam+1

In [36]:
# convert the ham and spam counts into frequencies
words.ham = words.ham/words.ham.sum()
words.spam = words.spam/words.spam.sum()
words.head()

,ham,spam
word,,
000,0.000062,0.003670
03,0.000062,0.002097
08000930705,0.000062,0.002272
10,0.000560,0.004020
100,0.000125,0.005767


In [37]:
# calculate the ratio of ham-to-spam and spam-to-ham for each word
words['ham_ratio'] = words.ham/words.spam
words['spam_ratio'] = words.spam/words.ham

In [38]:
# top 20 spam words
words.sort_values(by='spam_ratio', ascending=False).head(5)

,ham,spam,ham_ratio,spam_ratio
word,,,,
claim,0.000062,0.014156,0.004398,227.357742
prize,0.000062,0.013107,0.004750,210.516428
150p,0.000062,0.008738,0.007125,140.344285
guaranteed,0.000062,0.007340,0.008483,117.889200
tone,0.000062,0.006991,0.008907,112.275428


In [39]:
# top 20 ham words
words.sort_values(by='ham_ratio', ascending=False).head(5)

,ham,spam,ham_ratio,spam_ratio
word,,,,
gt,0.015939,0.000175,91.204284,0.010964
lt,0.015877,0.000175,90.848017,0.011007
lor,0.007783,0.000175,44.533342,0.022455
later,0.006662,0.000175,38.120540,0.026233
da,0.006538,0.000175,37.408007,0.026732


## Grid Search

In [51]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [55]:
vect = Pipeline(steps=[
               ('vect', CountVectorizer()),
               ('nb', MultinomialNB())])

In [60]:
param_dict = {
    'vect__stop_words': [None, 'english'],
    'vect__ngram_range':[(1,1),(1,2)],
    'vect__max_df':[1.0,0.9,0.8], # integer is a count, not a frequency
    'vect__min_df':[1,10,20,50],
    'vect__max_features' : [None, 500, 1000, 2000]
}

In [61]:
grid = GridSearchCV(vect,
                    param_dict,
                    cv=5,
                    scoring='accuracy'
)

In [62]:
grid

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vect', CountVectorizer()),
                                       ('nb', MultinomialNB())]),
             param_grid={'vect__max_df': [1.0, 0.9, 0.8],
                         'vect__max_features': [None, 500, 1000, 2000],
                         'vect__min_df': [1, 10, 20, 50],
                         'vect__ngram_range': [(1, 1), (1, 2)],
                         'vect__stop_words': [None, 'english']},
             scoring='accuracy')

In [63]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vect', CountVectorizer()),
                                       ('nb', MultinomialNB())]),
             param_grid={'vect__max_df': [1.0, 0.9, 0.8],
                         'vect__max_features': [None, 500, 1000, 2000],
                         'vect__min_df': [1, 10, 20, 50],
                         'vect__ngram_range': [(1, 1), (1, 2)],
                         'vect__stop_words': [None, 'english']},
             scoring='accuracy')

In [65]:
best = grid.best_estimator_

In [66]:
y_test_pred=best.predict(X_test)

In [67]:
# confusion matrix
confusion_matrix(y_test, y_test_pred)

array([[1192,    6],
       [  11,  184]])

## Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
log_clf = LogisticRegression()

In [ ]:
log_clf.fit(X_train_dtm,y_train)

In [ ]:
y_test_pred = log_clf.predict(X_test_dtm)

In [ ]:
accuracy_score(y_test,y_test_pred)

In [ ]:
confusion_matrix(y_test,y_test_pred)

In [ ]:
print(X_test[(y_test == 'spam') & (y_test_pred == 'ham')].iloc[5])

In [ ]:
# top coefficients
coeffs = pd.DataFrame(data = log_clf.coef_.T, index=vect.get_feature_names(),columns=['coefficient'])
coeffs.sort_values(by='coefficient').tail(20).plot.barh(figsize=(5,10))

In [ ]:
log_clf.classes_

# From occurrences to frequencies

Occurrence count is a good start but there is an issue: longer documents will have higher average count values than shorter documents, even though they might talk about the same topics.

To avoid these potential discrepancies it suffices to divide the number of occurrences of each word in a document by the total number of words in the document: these new features are called **tf (for Term Frequencies)**.

Another refinement on top of tf is to downscale weights for words that occur in many documents in the corpus and are therefore less informative than those that occur only in a smaller portion of the corpus. This downscaling is called **tf–idf (for “Term Frequency times Inverse Document Frequency”)**.

Both tf and tf–idf can be computed using [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer)

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:
# initialize
tf_transformer = TfidfTransformer(use_idf=False) # use tf
# fit
tf_transformer.fit(X_train_dtm)
# transform
X_train_tf = tf_transformer.transform(X_train_dtm)
X_test_tf = tf_transformer.transform(X_test_dtm)

In [ ]:
log_clf.fit(X_train_tf,y_train)
y_test_pred = log_clf.predict(X_test_tf)

In [ ]:
accuracy_score(y_test,y_test_pred)

In [ ]:
confusion_matrix(y_test,y_test_pred)